# 🌴 Aplikasi Prediksi Hasil Panen Kelapa Sawit

**Deskripsi:**  
Notebook ini digunakan untuk memprediksi hasil panen kelapa sawit (ton/ha) berdasarkan data yang diinputkan oleh pengguna menggunakan model Machine Learning.

**Fitur Input:**
- Umur Tanaman (tahun)
- Luas Lahan (ha)
- Curah Hujan (mm/bulan)
- Suhu Rata-rata (°C)
- Jumlah Pupuk (kg/ha)
- Jumlah Tenaga Kerja (orang/ha)
- Jenis Tanah
- Varietas Tanaman

---

## 📦 1. Instalasi & Import Library

In [ ]:
# Install library yang diperlukan
!pip install scikit-learn pandas numpy matplotlib seaborn ipywidgets -q

print("✅ Semua library berhasil diinstall!")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')

print("✅ Semua library berhasil diimport!")

## 📊 2. Pembuatan Dataset Sintetis & Pelatihan Model

In [ ]:
# ====================================================
# GENERATE DATA SINTETIS KELAPA SAWIT
# Berdasarkan karakteristik agronomis kelapa sawit
# ====================================================

np.random.seed(42)
n_samples = 1000

# --- Fitur Input ---
umur_tanaman     = np.random.uniform(3, 25, n_samples)          # tahun
luas_lahan       = np.random.uniform(0.5, 100, n_samples)       # ha
curah_hujan      = np.random.uniform(100, 400, n_samples)       # mm/bulan
suhu             = np.random.uniform(24, 34, n_samples)         # °C
jumlah_pupuk     = np.random.uniform(50, 500, n_samples)        # kg/ha
tenaga_kerja     = np.random.uniform(0.5, 5, n_samples)         # orang/ha

# Jenis Tanah
jenis_tanah_cat  = np.random.choice(['Gambut', 'Mineral', 'Latosol', 'Podsolik'], n_samples)
tanah_map        = {'Gambut': 0.85, 'Mineral': 1.0, 'Latosol': 1.1, 'Podsolik': 0.9}
tanah_faktor     = np.array([tanah_map[t] for t in jenis_tanah_cat])

# Varietas
varietas_cat     = np.random.choice(['Tenera', 'Dura', 'Pisifera', 'DxP Unggul'], n_samples)
varietas_map     = {'Tenera': 1.0, 'Dura': 0.85, 'Pisifera': 0.75, 'DxP Unggul': 1.15}
varietas_faktor  = np.array([varietas_map[v] for v in varietas_cat])

# --- Formula Hasil Panen (berbasis ilmu agronomi sawit) ---
# Fase produksi: mulai berproduksi usia 3 th, puncak 8-15 th, turun setelah 20 th
fase_umur = np.where(
    umur_tanaman < 3, 0,
    np.where(
        umur_tanaman <= 8,  (umur_tanaman - 3) / 5,
        np.where(
            umur_tanaman <= 15, 1.0,
            np.where(umur_tanaman <= 25, 1.0 - (umur_tanaman - 15) * 0.04, 0.5)
        )
    )
)

curah_faktor  = np.clip((curah_hujan - 100) / 300, 0.3, 1.0)
suhu_faktor   = np.where((suhu >= 26) & (suhu <= 32), 1.0,
                         np.where(suhu < 26, 0.85, 0.9))
pupuk_faktor  = np.clip(jumlah_pupuk / 250, 0.5, 1.2)
tk_faktor     = np.clip(tenaga_kerja / 2, 0.7, 1.1)

# Produksi TBS (ton/ha) — rata-rata nasional 15-25 ton/ha
hasil_panen = (
    20 * fase_umur * curah_faktor * suhu_faktor *
    pupuk_faktor * tk_faktor * tanah_faktor * varietas_faktor +
    np.random.normal(0, 1.0, n_samples)
)
hasil_panen = np.clip(hasil_panen, 0, 35)

# --- Buat DataFrame ---
df = pd.DataFrame({
    'Umur_Tanaman':   umur_tanaman,
    'Luas_Lahan':     luas_lahan,
    'Curah_Hujan':    curah_hujan,
    'Suhu':           suhu,
    'Jumlah_Pupuk':   jumlah_pupuk,
    'Tenaga_Kerja':   tenaga_kerja,
    'Jenis_Tanah':    jenis_tanah_cat,
    'Varietas':       varietas_cat,
    'Hasil_Panen':    hasil_panen
})

print("✅ Dataset berhasil dibuat!")
print(f"   Jumlah data  : {len(df)} sampel")
print(f"   Fitur        : {df.shape[1]-1} fitur")
print(f"\n📊 Statistik Hasil Panen (ton/ha):")
print(df['Hasil_Panen'].describe().round(2))

In [ ]:
# ====================================================
# PREPROCESSING & PELATIHAN MODEL
# ====================================================

# Label Encoding untuk fitur kategoris
le_tanah    = LabelEncoder()
le_varietas = LabelEncoder()

df_model = df.copy()
df_model['Jenis_Tanah'] = le_tanah.fit_transform(df['Jenis_Tanah'])
df_model['Varietas']    = le_varietas.fit_transform(df['Varietas'])

# Fitur dan target
feature_cols = ['Umur_Tanaman', 'Luas_Lahan', 'Curah_Hujan', 'Suhu',
                'Jumlah_Pupuk', 'Tenaga_Kerja', 'Jenis_Tanah', 'Varietas']

X = df_model[feature_cols]
y = df_model['Hasil_Panen']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# --- Latih beberapa model ---
models = {
    'Random Forest'         : RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting'     : GradientBoostingRegressor(n_estimators=200, random_state=42),
    'Ridge Regression'      : Ridge(alpha=1.0)
}

hasil_model = {}
print("🔄 Melatih model...\n")

for nama, model in models.items():
    if nama == 'Ridge Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    hasil_model[nama] = {'model': model, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'y_pred': y_pred}
    print(f"  ✅ {nama:25s} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.4f}")

# Pilih model terbaik (R² tertinggi)
best_name  = max(hasil_model, key=lambda k: hasil_model[k]['R2'])
best_model = hasil_model[best_name]['model']
print(f"\n🏆 Model terbaik: {best_name} (R² = {hasil_model[best_name]['R2']:.4f})")

## 📈 3. Visualisasi Performa Model

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('📊 Analisis Performa Model Prediksi Panen Sawit', fontsize=16, fontweight='bold', y=1.01)

colors = ['#2ecc71', '#3498db', '#e74c3c']

# --- Baris 1: Actual vs Predicted ---
for idx, (nama, info) in enumerate(hasil_model.items()):
    ax = axes[0, idx]
    ax.scatter(y_test, info['y_pred'], alpha=0.4, color=colors[idx], s=20)
    lims = [min(y_test.min(), info['y_pred'].min()), max(y_test.max(), info['y_pred'].max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Garis Ideal')
    ax.set_xlabel('Aktual (ton/ha)', fontsize=10)
    ax.set_ylabel('Prediksi (ton/ha)', fontsize=10)
    ax.set_title(f'{nama}\nR² = {info["R2"]:.4f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

# --- Baris 2 kiri: Perbandingan metrik ---
ax4 = axes[1, 0]
model_names = list(hasil_model.keys())
mae_vals    = [hasil_model[m]['MAE']  for m in model_names]
rmse_vals   = [hasil_model[m]['RMSE'] for m in model_names]
x_pos       = np.arange(len(model_names))
bars1 = ax4.bar(x_pos - 0.2, mae_vals,  0.35, label='MAE',  color='#3498db', alpha=0.8)
bars2 = ax4.bar(x_pos + 0.2, rmse_vals, 0.35, label='RMSE', color='#e74c3c', alpha=0.8)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(['RF', 'GB', 'Ridge'], fontsize=10)
ax4.set_ylabel('Error (ton/ha)', fontsize=10)
ax4.set_title('Perbandingan Error Model', fontsize=11, fontweight='bold')
ax4.legend(fontsize=9)
for bar in bars1: ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)
for bar in bars2: ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=8)

# --- Baris 2 tengah: Feature Importance ---
ax5 = axes[1, 1]
rf_model = hasil_model['Random Forest']['model']
importances = rf_model.feature_importances_
feat_names  = ['Umur\nTanaman', 'Luas\nLahan', 'Curah\nHujan', 'Suhu', 'Pupuk', 'T.Kerja', 'J.Tanah', 'Varietas']
sorted_idx  = np.argsort(importances)
ax5.barh(range(len(feat_names)), importances[sorted_idx], color='#2ecc71', alpha=0.8)
ax5.set_yticks(range(len(feat_names)))
ax5.set_yticklabels([feat_names[i] for i in sorted_idx], fontsize=9)
ax5.set_xlabel('Feature Importance', fontsize=10)
ax5.set_title('Feature Importance\n(Random Forest)', fontsize=11, fontweight='bold')

# --- Baris 2 kanan: Distribusi Hasil Panen ---
ax6 = axes[1, 2]
ax6.hist(df['Hasil_Panen'], bins=40, color='#9b59b6', alpha=0.7, edgecolor='white')
ax6.axvline(df['Hasil_Panen'].mean(), color='red', linestyle='--', linewidth=2, label=f'Rata-rata: {df["Hasil_Panen"].mean():.2f}')
ax6.set_xlabel('Hasil Panen (ton/ha)', fontsize=10)
ax6.set_ylabel('Frekuensi', fontsize=10)
ax6.set_title('Distribusi Hasil Panen\ndalam Dataset', fontsize=11, fontweight='bold')
ax6.legend(fontsize=9)

plt.tight_layout()
plt.savefig('performa_model.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Visualisasi performa model berhasil ditampilkan!")

## 🌴 4. Aplikasi Prediksi Interaktif

In [ ]:
# ====================================================
# APLIKASI PREDIKSI INTERAKTIF
# ====================================================

def prediksi_panen(umur, luas, hujan, suhu_val, pupuk, tk, tanah, varietas):
    """Fungsi prediksi menggunakan model terbaik."""
    tanah_enc    = le_tanah.transform([tanah])[0]
    varietas_enc = le_varietas.transform([varietas])[0]
    X_input      = np.array([[umur, luas, hujan, suhu_val, pupuk, tk, tanah_enc, varietas_enc]])

    # Prediksi dari semua model
    pred_rf = hasil_model['Random Forest']['model'].predict(X_input)[0]
    pred_gb = hasil_model['Gradient Boosting']['model'].predict(X_input)[0]

    X_scaled = scaler.transform(X_input)
    pred_rd  = hasil_model['Ridge Regression']['model'].predict(X_scaled)[0]

    # Prediksi ensemble (rata-rata terbobot)
    r2_vals   = [hasil_model[m]['R2'] for m in ['Random Forest', 'Gradient Boosting', 'Ridge Regression']]
    total_r2  = sum(r2_vals)
    bobot     = [r/total_r2 for r in r2_vals]
    pred_ens  = bobot[0]*pred_rf + bobot[1]*pred_gb + bobot[2]*pred_rd

    return max(0, pred_rf), max(0, pred_gb), max(0, pred_rd), max(0, pred_ens)


def kategori_hasil(nilai):
    if nilai >= 22:   return "🟢 SANGAT TINGGI", '#27ae60'
    elif nilai >= 18: return "🟡 TINGGI",        '#f39c12'
    elif nilai >= 13: return "🟠 SEDANG",        '#e67e22'
    else:             return "🔴 RENDAH",         '#e74c3c'


def rekomendasi_otomatis(umur, hujan, suhu_val, pupuk, tk):
    saran = []
    if umur < 3:
        saran.append("⚠️ Tanaman belum memasuki fase produksi (butuh min. 3 tahun).")
    elif umur > 20:
        saran.append("⚠️ Tanaman sudah memasuki fase penuaan. Pertimbangkan peremajaan.")
    if hujan < 150:
        saran.append("💧 Curah hujan kurang — pertimbangkan irigasi tambahan.")
    elif hujan > 350:
        saran.append("🌧️ Curah hujan berlebihan — pastikan drainase lahan memadai.")
    if suhu_val < 26:
        saran.append("🌡️ Suhu terlalu rendah — berpotensi menghambat pertumbuhan.")
    elif suhu_val > 32:
        saran.append("🔥 Suhu terlalu tinggi — risiko stres tanaman meningkat.")
    if pupuk < 150:
        saran.append("🌱 Dosis pupuk sangat rendah — tingkatkan pemupukan sesuai rekomendasi.")
    elif pupuk < 250:
        saran.append("🌱 Pertimbangkan meningkatkan dosis pupuk untuk hasil optimal.")
    if tk < 1.0:
        saran.append("👷 Tenaga kerja kurang — panen bisa terlambat dan memengaruhi kualitas.")
    if not saran:
        saran.append("✅ Kondisi lahan sudah optimal! Pertahankan praktik budidaya.")
    return saran


# ==== BUAT WIDGET ====
style_label = {'description_width': '160px'}
layout_w    = widgets.Layout(width='420px')

w_umur     = widgets.FloatSlider(value=8,  min=1,  max=25,  step=0.5,  description='Umur Tanaman (th):',   style=style_label, layout=layout_w, readout_format='.1f')
w_luas     = widgets.FloatSlider(value=10, min=0.5,max=100, step=0.5,  description='Luas Lahan (ha):',     style=style_label, layout=layout_w, readout_format='.1f')
w_hujan    = widgets.FloatSlider(value=200,min=50, max=450, step=5,    description='Curah Hujan (mm/bln):',style=style_label, layout=layout_w, readout_format='.0f')
w_suhu     = widgets.FloatSlider(value=28, min=22, max=38,  step=0.5,  description='Suhu Rata-rata (°C):',  style=style_label, layout=layout_w, readout_format='.1f')
w_pupuk    = widgets.FloatSlider(value=250,min=50, max=600, step=10,   description='Jumlah Pupuk (kg/ha):', style=style_label, layout=layout_w, readout_format='.0f')
w_tk       = widgets.FloatSlider(value=2,  min=0.5,max=6,   step=0.1,  description='Tenaga Kerja (org/ha):',style=style_label,layout=layout_w, readout_format='.1f')
w_tanah    = widgets.Dropdown(options=['Mineral','Gambut','Latosol','Podsolik'], value='Mineral', description='Jenis Tanah:',     style=style_label, layout=layout_w)
w_varietas = widgets.Dropdown(options=['Tenera','DxP Unggul','Dura','Pisifera'],  value='Tenera',  description='Varietas:',         style=style_label, layout=layout_w)

btn_prediksi = widgets.Button(description='🌴 PREDIKSI SEKARANG', button_style='success',
                               layout=widgets.Layout(width='250px', height='45px'))
btn_reset    = widgets.Button(description='🔄 Reset',             button_style='warning',
                               layout=widgets.Layout(width='120px', height='45px'))

output_area = widgets.Output()


def on_prediksi(b):
    with output_area:
        clear_output(wait=True)
        rf, gb, rd, ens = prediksi_panen(
            w_umur.value, w_luas.value, w_hujan.value, w_suhu.value,
            w_pupuk.value, w_tk.value, w_tanah.value, w_varietas.value
        )

        kat, warna = kategori_hasil(ens)
        total_prod  = ens * w_luas.value
        saran_list  = rekomendasi_otomatis(w_umur.value, w_hujan.value, w_suhu.value, w_pupuk.value, w_tk.value)

        # --- Tabel hasil ---
        html_hasil = f"""
        <div style="font-family:Arial; border:2px solid {warna}; border-radius:12px; padding:20px; max-width:680px; background:#fafafa;">
          <h2 style="color:#2c3e50; text-align:center; margin-bottom:16px;">🌴 Hasil Prediksi Panen Sawit</h2>

          <div style="background:{warna}; color:white; border-radius:8px; padding:14px; text-align:center; margin-bottom:16px;">
            <span style="font-size:28px; font-weight:bold;">{ens:.2f} ton/ha</span><br>
            <span style="font-size:16px;">{kat}</span>
          </div>

          <table style="width:100%; border-collapse:collapse; font-size:14px;">
            <tr style="background:#ecf0f1;">
              <th style="padding:8px; text-align:left; border:1px solid #ddd;">Model</th>
              <th style="padding:8px; text-align:center; border:1px solid #ddd;">Prediksi (ton/ha)</th>
              <th style="padding:8px; text-align:center; border:1px solid #ddd;">Akurasi (R²)</th>
            </tr>
            <tr>
              <td style="padding:8px; border:1px solid #ddd;">🌳 Random Forest</td>
              <td style="padding:8px; text-align:center; border:1px solid #ddd; font-weight:bold;">{rf:.2f}</td>
              <td style="padding:8px; text-align:center; border:1px solid #ddd;">{hasil_model['Random Forest']['R2']:.4f}</td>
            </tr>
            <tr style="background:#f9f9f9;">
              <td style="padding:8px; border:1px solid #ddd;">📈 Gradient Boosting</td>
              <td style="padding:8px; text-align:center; border:1px solid #ddd; font-weight:bold;">{gb:.2f}</td>
              <td style="padding:8px; text-align:center; border:1px solid #ddd;">{hasil_model['Gradient Boosting']['R2']:.4f}</td>
            </tr>
            <tr>
              <td style="padding:8px; border:1px solid #ddd;">📐 Ridge Regression</td>
              <td style="padding:8px; text-align:center; border:1px solid #ddd; font-weight:bold;">{rd:.2f}</td>
              <td style="padding:8px; text-align:center; border:1px solid #ddd;">{hasil_model['Ridge Regression']['R2']:.4f}</td>
            </tr>
            <tr style="background:#e8f8f5;">
              <td style="padding:8px; border:1px solid #ddd; font-weight:bold;">🏆 Ensemble (Rata-rata Terbobot)</td>
              <td style="padding:8px; text-align:center; border:1px solid #ddd; font-weight:bold; color:{warna}; font-size:16px;">{ens:.2f}</td>
              <td style="padding:8px; text-align:center; border:1px solid #ddd;">—</td>
            </tr>
          </table>

          <div style="margin-top:16px; padding:12px; background:#eaf4fb; border-radius:8px; border-left:4px solid #3498db;">
            <strong>📦 Estimasi Total Produksi:</strong><br>
            {ens:.2f} ton/ha × {w_luas.value:.1f} ha = <strong style="font-size:18px;">{total_prod:.1f} ton TBS</strong><br>
            <span style="font-size:12px; color:#666;">*TBS = Tandan Buah Segar</span>
          </div>

          <div style="margin-top:16px; padding:12px; background:#fef9e7; border-radius:8px; border-left:4px solid #f39c12;">
            <strong>💡 Rekomendasi:</strong><br>
            {'<br>'.join(['• ' + s for s in saran_list])}
          </div>
        </div>
        """
        display(HTML(html_hasil))

        # --- Chart bar prediksi per model ---
        fig2, ax = plt.subplots(figsize=(8, 3.5))
        model_labels = ['Random\nForest', 'Gradient\nBoosting', 'Ridge\nRegression', 'Ensemble']
        pred_vals    = [rf, gb, rd, ens]
        bar_colors   = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']
        bars = ax.bar(model_labels, pred_vals, color=bar_colors, alpha=0.85, width=0.5, edgecolor='white', linewidth=1.5)
        for bar, val in zip(bars, pred_vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.15, f'{val:.2f}',
                    ha='center', va='bottom', fontweight='bold', fontsize=11)
        ax.set_ylabel('Prediksi Hasil Panen (ton/ha)', fontsize=11)
        ax.set_title('Perbandingan Prediksi Antar Model', fontsize=12, fontweight='bold')
        ax.set_ylim(0, max(pred_vals)*1.25 + 1)
        ax.axhline(y=20, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Rata-rata Nasional (~20 ton/ha)')
        ax.legend(fontsize=9)
        plt.tight_layout()
        plt.show()


def on_reset(b):
    w_umur.value=8; w_luas.value=10; w_hujan.value=200; w_suhu.value=28
    w_pupuk.value=250; w_tk.value=2; w_tanah.value='Mineral'; w_varietas.value='Tenera'
    with output_area:
        clear_output()


btn_prediksi.on_click(on_prediksi)
btn_reset.on_click(on_reset)

# ==== LAYOUT UI ====
header_html = widgets.HTML("""
<div style="font-family:Arial; background:linear-gradient(135deg,#27ae60,#2ecc71); color:white;
     padding:20px; border-radius:12px; margin-bottom:16px; max-width:680px;">
  <h2 style="margin:0; font-size:22px;">🌴 Aplikasi Prediksi Panen Kelapa Sawit</h2>
  <p style="margin:6px 0 0; font-size:13px; opacity:0.9;">Masukkan data lahan Anda untuk mendapatkan estimasi hasil panen TBS (Tandan Buah Segar)</p>
</div>
""")

section1 = widgets.HTML("<b style='color:#2c3e50; font-size:14px;'>📋 Data Lahan & Tanaman</b>")
section2 = widgets.HTML("<b style='color:#2c3e50; font-size:14px;'>🌦️ Data Iklim & Manajemen</b>")

col1 = widgets.VBox([section1, w_umur, w_luas, w_tanah, w_varietas],
                     layout=widgets.Layout(margin='0 20px 0 0'))
col2 = widgets.VBox([section2, w_hujan, w_suhu, w_pupuk, w_tk])

row_inputs = widgets.HBox([col1, col2])
row_btns   = widgets.HBox([btn_prediksi, btn_reset], layout=widgets.Layout(margin='16px 0'))

app = widgets.VBox([header_html, row_inputs, row_btns, output_area])
display(app)

## 📥 5. Prediksi dari File CSV (Batch)

In [ ]:
# ====================================================
# PREDIKSI BATCH DARI FILE CSV
# ====================================================

# Buat file CSV contoh
contoh_csv = pd.DataFrame({
    'Umur_Tanaman':  [5, 10, 15, 3, 22],
    'Luas_Lahan':    [5, 20, 10, 50, 8],
    'Curah_Hujan':   [200, 250, 180, 300, 150],
    'Suhu':          [28, 27, 30, 26, 32],
    'Jumlah_Pupuk':  [200, 350, 150, 400, 100],
    'Tenaga_Kerja':  [2, 3, 1.5, 4, 1],
    'Jenis_Tanah':   ['Mineral', 'Latosol', 'Gambut', 'Mineral', 'Podsolik'],
    'Varietas':      ['Tenera', 'DxP Unggul', 'Tenera', 'Dura', 'Tenera']
})
contoh_csv.to_csv('data_lahan_contoh.csv', index=False)
print("✅ File contoh 'data_lahan_contoh.csv' berhasil dibuat!")


def prediksi_dari_csv(filepath):
    """Baca CSV dan prediksi semua baris."""
    df_input = pd.read_csv(filepath)

    # Encode kategoris
    df_input['Jenis_Tanah'] = le_tanah.transform(df_input['Jenis_Tanah'])
    df_input['Varietas']    = le_varietas.transform(df_input['Varietas'])

    X_in     = df_input[feature_cols]
    X_scaled = scaler.transform(X_in)

    pred_rf  = hasil_model['Random Forest']['model'].predict(X_in)
    pred_gb  = hasil_model['Gradient Boosting']['model'].predict(X_in)
    pred_rd  = hasil_model['Ridge Regression']['model'].predict(X_scaled)

    r2_vals = [hasil_model[m]['R2'] for m in ['Random Forest', 'Gradient Boosting', 'Ridge Regression']]
    total_r2 = sum(r2_vals)
    bobot    = [r/total_r2 for r in r2_vals]
    pred_ens = bobot[0]*pred_rf + bobot[1]*pred_gb + bobot[2]*pred_rd

    df_out = pd.read_csv(filepath)
    df_out['Pred_RF']       = np.round(np.maximum(pred_rf, 0), 2)
    df_out['Pred_GB']       = np.round(np.maximum(pred_gb, 0), 2)
    df_out['Pred_Ridge']    = np.round(np.maximum(pred_rd, 0), 2)
    df_out['Pred_Ensemble'] = np.round(np.maximum(pred_ens, 0), 2)
    df_out['Kategori']      = df_out['Pred_Ensemble'].apply(
        lambda x: 'SANGAT TINGGI' if x>=22 else ('TINGGI' if x>=18 else ('SEDANG' if x>=13 else 'RENDAH'))
    )
    df_out['Total_Produksi_Ton'] = np.round(df_out['Pred_Ensemble'] * df_out['Luas_Lahan'], 2)
    return df_out


# Jalankan prediksi batch
df_hasil_batch = prediksi_dari_csv('data_lahan_contoh.csv')

print("\n📋 Hasil Prediksi Batch:")
display(df_hasil_batch[['Umur_Tanaman','Luas_Lahan','Pred_RF','Pred_GB','Pred_Ridge',
                         'Pred_Ensemble','Kategori','Total_Produksi_Ton']].style
        .background_gradient(subset=['Pred_Ensemble'], cmap='YlGn')
        .format({'Pred_RF':'{:.2f}','Pred_GB':'{:.2f}','Pred_Ridge':'{:.2f}',
                 'Pred_Ensemble':'{:.2f}','Total_Produksi_Ton':'{:.1f}'})
)

# Simpan hasil
df_hasil_batch.to_csv('hasil_prediksi_batch.csv', index=False)
print("\n✅ Hasil disimpan ke 'hasil_prediksi_batch.csv'")
print(f"   Total estimasi produksi: {df_hasil_batch['Total_Produksi_Ton'].sum():.1f} ton TBS")

## 📤 6. Upload & Prediksi CSV Anda Sendiri

In [ ]:
# ====================================================
# UPLOAD FILE CSV DARI PERANGKAT ANDA
# ====================================================

from google.colab import files as colab_files
import io

print("📤 Klik tombol di bawah untuk upload file CSV Anda.")
print("\n📌 Format kolom CSV yang diperlukan:")
print("   Umur_Tanaman | Luas_Lahan | Curah_Hujan | Suhu | Jumlah_Pupuk | Tenaga_Kerja | Jenis_Tanah | Varietas")
print("\n   Jenis_Tanah : Mineral / Gambut / Latosol / Podsolik")
print("   Varietas    : Tenera / DxP Unggul / Dura / Pisifera")

uploaded = colab_files.upload()

for filename, content in uploaded.items():
    print(f"\n✅ File '{filename}' berhasil diupload!")
    df_upload = pd.read_csv(io.StringIO(content.decode('utf-8')))

    df_upload['Jenis_Tanah'] = le_tanah.transform(df_upload['Jenis_Tanah'])
    df_upload['Varietas']    = le_varietas.transform(df_upload['Varietas'])

    X_up     = df_upload[feature_cols]
    X_scaled = scaler.transform(X_up)

    pred_rf  = hasil_model['Random Forest']['model'].predict(X_up)
    pred_gb  = hasil_model['Gradient Boosting']['model'].predict(X_up)
    pred_rd  = hasil_model['Ridge Regression']['model'].predict(X_scaled)

    r2_vals  = [hasil_model[m]['R2'] for m in ['Random Forest','Gradient Boosting','Ridge Regression']]
    total_r2 = sum(r2_vals)
    bobot    = [r/total_r2 for r in r2_vals]
    pred_ens = np.maximum(bobot[0]*pred_rf + bobot[1]*pred_gb + bobot[2]*pred_rd, 0)

    df_out_up = pd.read_csv(io.StringIO(content.decode('utf-8')))
    df_out_up['Prediksi_Hasil_Panen_tonha'] = np.round(pred_ens, 2)
    df_out_up['Total_Produksi_Ton'] = np.round(pred_ens * df_out_up['Luas_Lahan'], 2)

    output_filename = 'hasil_' + filename
    df_out_up.to_csv(output_filename, index=False)

    print(f"\n📋 Hasil Prediksi (5 baris pertama):")
    display(df_out_up.head())
    print(f"\n💾 Mengunduh '{output_filename}'...")
    colab_files.download(output_filename)

## 📊 7. Analisis Sensitivitas Faktor

In [ ]:
# ====================================================
# ANALISIS SENSITIVITAS — Dampak tiap faktor terhadap hasil
# ====================================================

# Nilai default (kondisi optimal)
default_vals = {
    'Umur_Tanaman': 10, 'Luas_Lahan': 10, 'Curah_Hujan': 220,
    'Suhu': 28, 'Jumlah_Pupuk': 300, 'Tenaga_Kerja': 2.5,
    'Jenis_Tanah': le_tanah.transform(['Mineral'])[0],
    'Varietas': le_varietas.transform(['Tenera'])[0]
}

analisis_config = {
    'Umur Tanaman (th)':      ('Umur_Tanaman',  np.arange(3, 26, 1)),
    'Curah Hujan (mm/bln)':   ('Curah_Hujan',   np.arange(80, 420, 20)),
    'Suhu (°C)':              ('Suhu',           np.arange(22, 38, 0.5)),
    'Jumlah Pupuk (kg/ha)':   ('Jumlah_Pupuk',  np.arange(50, 600, 25)),
    'Tenaga Kerja (org/ha)':  ('Tenaga_Kerja',   np.arange(0.5, 6, 0.2)),
}

fig3, axes3 = plt.subplots(2, 3, figsize=(18, 10))
fig3.suptitle('🔬 Analisis Sensitivitas — Pengaruh Faktor terhadap Hasil Panen', fontsize=15, fontweight='bold')

ax_list   = axes3.flatten()
pal_lines = ['#e74c3c', '#3498db', '#27ae60', '#f39c12', '#9b59b6']

for idx, (label, (kolom, rentang)) in enumerate(analisis_config.items()):
    preds = []
    for val in rentang:
        row = [default_vals[f] for f in feature_cols]
        col_idx = feature_cols.index(kolom)
        row[col_idx] = val
        X_in = np.array([row])
        pred = hasil_model['Random Forest']['model'].predict(X_in)[0]
        preds.append(max(0, pred))

    ax = ax_list[idx]
    ax.plot(rentang, preds, color=pal_lines[idx], linewidth=2.5)
    ax.fill_between(rentang, preds, alpha=0.15, color=pal_lines[idx])
    ax.axhline(y=np.mean(preds), color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Prediksi Panen (ton/ha)', fontsize=10)
    ax.set_title(f'Pengaruh {label}', fontsize=11, fontweight='bold')
    ax.set_ylim(bottom=0)

# Sembunyikan subplot kosong
ax_list[-1].axis('off')

plt.tight_layout()
plt.savefig('analisis_sensitivitas.png', dpi=120, bbox_inches='tight')
plt.show()
print("✅ Analisis sensitivitas selesai!")

---
## ℹ️ Catatan Penggunaan

| Kolom | Satuan | Nilai Tipikal |
|---|---|---|
| Umur Tanaman | Tahun | 3–25 |
| Luas Lahan | Ha | Bebas |
| Curah Hujan | mm/bulan | 150–300 |
| Suhu | °C | 26–32 |
| Jumlah Pupuk | kg/ha/tahun | 150–400 |
| Tenaga Kerja | orang/ha | 1–4 |
| Jenis Tanah | - | Mineral/Gambut/Latosol/Podsolik |
| Varietas | - | Tenera/DxP Unggul/Dura/Pisifera |

> ⚠️ **Disclaimer:** Model ini menggunakan data sintetis berbasis parameter agronomis. Untuk akurasi lebih tinggi, latih ulang model menggunakan data historis lahan Anda sendiri.

---
*Dibuat untuk kebutuhan analisis pertanian kelapa sawit Indonesia 🇮🇩*